# visu-predict — Colab launcher

End-to-end training of a `visu-predict` model from Google Colab. Covers the legacy `TrafficTransformer` path and the full SOTA upgrade stack (Tier 1–3 PRs): adaptive embedding, alternating spatial↔temporal attention, learnable adjacency, PatchTST patching, masked-reconstruction pretraining, optional Mamba/SSM temporal block, and identity/adaptive mixture-of-experts.

Sections in this notebook:

1. Clone + install
2. Smoke verify
3. (optional) Drive mount + data download
4. Choose a config preset
5. Train
6. Pretrain → fine-tune workflow
7. Optional: Mamba temporal block
8. Notes (memory, MoE prerequisites)

Repo: <https://github.com/almo-intellect/visu-predict>

## 1. Clone + install

While the SOTA upgrade work lives on the `sota-upgrade` branch (and not yet on `main`), keep `BRANCH = 'sota-upgrade'`. Switch to `'main'` once the integration PR has been promoted.

In [ ]:
BRANCH = "sota-upgrade"  # 'main' once Tier 1-3 is promoted; any PR branch to test

!git clone --branch {BRANCH} --depth 1 https://github.com/almo-intellect/visu-predict.git
%cd visu-predict

In [ ]:
# Install the package with the GNN + holidays extras. Add 'mamba' to enable
# the optional Mamba temporal block (CUDA only — install in a later cell instead).
!pip install -q -e ".[gnn,holidays]"

## 2. Smoke verify

Confirm the package imports cleanly and exposes the SOTA-stack modules before spending time on training.

In [ ]:
import visu_predict
from visu_predict.config import TrainingConfig
from visu_predict.models.transformer import TrafficTransformer
from visu_predict.models.embeddings import STAEInputComposer
from visu_predict.models.st_blocks import STAttnStack
from visu_predict.models.adaptive_graph import AdaptiveAdjacency
from visu_predict.models.patching import PatchEmbed
from visu_predict.models.moe import STMoE
from visu_predict.training.pretrain import pretrain, MaskedSTDataset
from visu_predict.models.mamba_block import MAMBA_AVAILABLE

print("visu_predict", visu_predict.__version__)
print("mamba available:", MAMBA_AVAILABLE)
import torch
print("cuda available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

## 3. (optional) Drive mount + data download

Two options:
- **Drive mount**: keep checkpoints and downloads in your Google Drive across Colab sessions.
- **Download script**: pull the maintainer's shared Drive folder into Colab's local disk via `gdown` (lost when the session ends).

In [ ]:
# Option A: mount Drive (recommended — survives session restarts).
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/visu-predict/inputs'
OUTPUT_ROOT = '/content/drive/MyDrive/visu-predict/outputs'
print('data ->', DATA_ROOT)
print('outputs ->', OUTPUT_ROOT)

In [ ]:
# Option B: pull the shared dataset folder. Skip if files are already in Drive.
!pip install -q gdown
!python scripts/download_data.py --dest {DATA_ROOT}

## 4. Choose a config preset

Edit `configs/example.yaml` in place. Three presets covered below — pick one:

1. **Legacy** — the original `TrafficTransformer`. Default in `configs/example.yaml`, no edits needed.
2. **STAE core (Tier 1)** — adaptive embedding + alternating spatial/temporal attention + adaptive adjacency.
3. **Full SOTA (Tier 1+2+3)** — STAE core + temporal patching + MoE. Strongest recipe.

The `d_*` dims (`d_input + d_tod + d_dow + d_adaptive + d_node`) must sum to `hidden_dim`.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/example.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

# Always redirect outputs into Drive so checkpoints survive session restarts.
cfg['base_output_dir'] = OUTPUT_ROOT

PRESET = 'stae_core'  # one of: 'legacy', 'stae_core', 'full_sota'

if PRESET == 'stae_core':
    cfg.update({
        'model_pipeline': 'stae',
        'use_discrete_time_embeddings': True,
        'steps_per_day': 288,                # 5-minute METR-LA / PEMS-BAY
        'hidden_dim': 96,                    # smaller for Colab T4
        'num_heads': 8,
        'd_input': 24, 'd_tod': 24, 'd_dow': 24, 'd_adaptive': 24, 'd_node': 0,
        'interleave_order': 'TS',
        'use_adaptive_adjacency': True,
        'adaptive_adj_dim': 10,
        'adaptive_adj_inject_into': 'spatial_attn',
    })

elif PRESET == 'full_sota':
    cfg.update({
        'model_pipeline': 'stae',
        'use_discrete_time_embeddings': True,
        'steps_per_day': 288,
        'hidden_dim': 96,
        'num_heads': 8,
        'd_input': 24, 'd_tod': 24, 'd_dow': 24, 'd_adaptive': 24, 'd_node': 0,
        'interleave_order': 'TS',
        'use_adaptive_adjacency': True,
        'adaptive_adj_dim': 10,
        'adaptive_adj_inject_into': 'spatial_attn',
        'use_temporal_patching': True,
        'patch_length': 4,
        'use_moe': True,                     # requires adaptive adjacency
        'moe_load_balance_weight': 0.01,
    })
    # MoE's identity expert reads the static .pkl adjacency. Make sure it's
    # present in DATA_ROOT (e.g. adj_METR-LA.pkl) or train without --use_moe.
    cfg.setdefault('use_spatial_features', True)

# 'legacy' falls through: keep the file as-is.

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump({k: v for k, v in cfg.items() if k.startswith(('model_', 'use_', 'd_', 'hidden_', 'num_'))}, sort_keys=False))

## 5. Train

Run the CLI. Replace `METR-LA.csv` with whichever traffic CSV you want.

In [ ]:
DATA_CSV = f'{DATA_ROOT}/METR-LA.csv'

!visu-predict train --config configs/example.yaml --data {DATA_CSV}

## 6. Pretrain → fine-tune workflow

The STAE pipeline supports self-supervised masked-reconstruction pretraining (PR #6). Pretrain on one dataset, then fine-tune the encoder on a target dataset's forecasting task.

Requires `model_pipeline: stae` in the config.

In [ ]:
# Step 1 — pretrain. Writes <output_dir>/pretrained/encoder.pth.
PRETRAIN_DATA = f'{DATA_ROOT}/METR-LA.csv'   # source dataset for self-supervised pretraining
!visu-predict pretrain --config configs/example.yaml --data {PRETRAIN_DATA}

In [ ]:
# Step 2 — locate the pretrained encoder, point the config at it, fine-tune on the target dataset.
import glob, os
encoder_paths = sorted(glob.glob(f'{OUTPUT_ROOT}/*/pretrained/encoder.pth'))
assert encoder_paths, 'no encoder.pth found — did the pretrain step finish?'
encoder_path = encoder_paths[-1]
print('using', encoder_path)

cfg = yaml.safe_load(cfg_path.read_text())
cfg['pretrained_encoder_path'] = encoder_path
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

FINE_TUNE_DATA = f'{DATA_ROOT}/PEMS-BAY.csv'   # target dataset (could be the same)
!visu-predict train --config configs/example.yaml --data {FINE_TUNE_DATA}

## 7. Optional: Mamba/SSM temporal block

`mamba-ssm` is a CUDA-only optional dependency. Install it before flipping `temporal_block_type` to `'mamba'`. Don't bother on a CPU runtime — the install will fail or the Mamba block will refuse to run.

In [ ]:
import torch
if torch.cuda.is_available():
    !pip install -q causal-conv1d>=1.2.0
    !pip install -q mamba-ssm>=2.0
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg['temporal_block_type'] = 'mamba'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Mamba enabled. Re-run section 5 to train.')
else:
    print('CUDA not available; skipping Mamba install. Use the default attention temporal block.')

## 8. Notes

**Compute budget.** With `d_adaptive=80` and the default Tier 1 config, the adaptive embedding alone is `steps_per_day × num_sensors × d_adp` parameters: ~20 M for PEMS-07 (882 sensors). A T4 (16 GB) handles this; large batch sizes or LargeST-scale graphs need an A100. The presets above use `d_adaptive=24` to stay comfortable on Colab free-tier T4 / L4.

**MoE prerequisites.** `use_moe=True` requires `use_adaptive_adjacency=True` (the adaptive expert reads from it). The identity expert reads a static `[N, N]` adjacency that the runner pulls from `<input_dir>/adj_<dataset>.pkl` when `use_spatial_features=True`. If the `.pkl` is absent, the identity expert silently uses an identity matrix — the MoE will still train but degenerate toward the adaptive expert.

**Adaptive adjacency injection.** `adaptive_adj_inject_into` controls who sees the learned graph: `'spatial_attn'` adds it as an attention bias in STAE's `SpatialBlock`; `'gnn'` feeds it as the adjacency for the legacy `GCNEncoder` (mutually exclusive with .pkl); `'both'` does both at once.

**Mixed precision.** `use_mixed_precision: true` is the default and cuts memory ~2×. The adaptive adjacency softmax is wrapped in `autocast(enabled=False)` internally to avoid NaNs on >500-sensor graphs.

**Validating Tier 1 against published SOTA.** STAEformer's headline numbers on METR-LA (h=12): MAE ≈ 3.34, RMSE ≈ 6.83, MAPE ≈ 9.40%. Reproduce by setting `seq_length: 12`, `pred_length: 12`, `hidden_dim: 96`, `d_adaptive: 24`, `interleave_order: 'TS'`, `num_layers: 3`, training on METR-LA with the default Adam + plateau scheduler. Expect ~1 hr on a T4.